# ClipForge on Google Colab (free GPU)

Use this when your laptop is too slow for a full 2-3 hour episode. Colab's free **T4 GPU** runs Whisper `large-v3`
about 10× faster than a laptop CPU. Clip selection uses a **free** Gemini or Groq API key (Ollama is not needed here).

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Steps: run each cell top to bottom. Output (mp4 clips + metadata.csv) is zipped to your Google Drive.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Install ClipForge (takes ~2 min)
!git clone -q https://github.com/hamsidhi/clipforge.git /content/clipforge || (cd /content/clipforge && git pull -q)
%cd /content/clipforge
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install -r requirements.txt
!pip -q install -e . --no-deps
!clipforge doctor --fix || true

In [ ]:
#@title 3. API key (free) — Gemini recommended
#@markdown Get a free key at https://aistudio.google.com/apikey (Gemini) or https://console.groq.com/keys (Groq).
provider = "gemini" #@param ["gemini", "groq"]
api_key = "" #@param {type:"string"}
import os
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY" if provider == "gemini" else "GROQ_API_KEY")
    except Exception:
        pass
assert api_key, "Paste your free API key above (or store it in Colab Secrets)."
with open(".env", "w") as f:
    f.write(f"{'GEMINI_API_KEY' if provider == 'gemini' else 'GROQ_API_KEY'}={api_key}
")
print("Key saved to .env (only inside this Colab session).")

In [ ]:
#@title 4. Mount Google Drive (outputs are saved there)
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/ClipForge"
os.makedirs(OUT, exist_ok=True)

In [ ]:
#@title 5. Run ClipForge
url = "https://www.youtube.com/watch?v=..." #@param {type:"string"}
clips = 20 #@param {type:"slider", min:15, max:25, step:1}
caption_script = "roman" #@param ["roman", "devanagari"]
layout = "auto" #@param ["auto", "track", "split", "blur"]
# Whisper picks large-v3 automatically on the GPU.
!clipforge run "{url}" --clips {clips} --provider {provider} --caption-script {caption_script} --layout {layout}

In [ ]:
#@title 6. Zip the clips + metadata to Google Drive
import glob, shutil, json
vids = sorted(glob.glob("workspace/*/clips"), key=os.path.getmtime)
assert vids, "No clips found — check the output of step 5."
ws = os.path.dirname(vids[-1])
vid = os.path.basename(ws)
staging = f"/content/export_{vid}"
shutil.rmtree(staging, ignore_errors=True)
os.makedirs(staging)
for f in glob.glob(f"{ws}/clips/*.mp4"):
    shutil.copy(f, staging)
for f in ("metadata.csv", "metadata.json", "gallery.html", "candidates.json"):
    if os.path.exists(f"{ws}/{f}"):
        shutil.copy(f"{ws}/{f}", staging)
zip_path = shutil.make_archive(f"{OUT}/clipforge_{vid}", "zip", staging)
print("Saved:", zip_path)

In [ ]:
#@title 7. (Optional) Preview clips here
from IPython.display import HTML, display
from base64 import b64encode
for f in sorted(glob.glob(f"{ws}/clips/*.mp4"))[:6]:
    data = b64encode(open(f, "rb").read()).decode()
    display(HTML(f'<div style="display:inline-block;margin:6px"><p>{os.path.basename(f)}</p>'
                 f'<video width=220 controls src="data:video/mp4;base64,{data}"></video></div>'))

### Tips
- **Re-run is cheap:** every stage is cached in `workspace/<video_id>/`. If Colab disconnects, re-run step 5 — finished stages are skipped (as long as the runtime wasn't reset).
- To keep the cache across sessions, set `output.dir` in `config.yaml` to a Drive folder, e.g. `/content/drive/MyDrive/ClipForge/workspace`.
- To edit clips: change `workspace/<id>/candidates.json`, then `!clipforge render <id> --only 3`.